# Automated Detection and Grading of Diabetic Retinopathy Using CNN

This notebook walks through the full pipeline end to end, following the seven
stages of the system architecture. It is a guided tour of the code in `src/`,
not a reimplementation - every cell calls the same functions the command-line
pipeline uses, so results here and from `make all` are identical.

| Stage | What happens |
|---|---|
| 1 | Retinal image dataset |
| 2 | Image preprocessing |
| 3 | Training / validation split |
| 4 | CNN model |
| 5 | Feature extraction |
| 6 | Classification |
| 7 | DR grade / prediction |

In [ ]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from src import config as C

print("project root :", ROOT)
print("torch        :", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

## Stage 1 - the dataset

APTOS 2019: 3,662 colour fundus photographs from Aravind Eye Hospital, each
graded 0-4 by a clinician. If `train.csv` is missing, run
`python -m src.download_data` from the project root first.

In [ ]:
df = pd.read_csv(C.LABELS_CSV)
print(f"{len(df)} labelled images\n")

counts = df["diagnosis"].value_counts().sort_index()
summary = pd.DataFrame({
    "grade": counts.index,
    "name": [C.CLASS_NAMES[i] for i in counts.index],
    "images": counts.values,
    "share": (counts.values / len(df) * 100).round(1),
})
summary

Nearly half the dataset is grade 0. A model that predicts "No DR" for every
image would score ~49% accuracy while being clinically useless - which is
exactly why we weight the loss by inverse class frequency and report
quadratic weighted kappa rather than accuracy.

In [ ]:
from src.eda import grade_distribution
grade_distribution(df)
plt.close("all")

from IPython.display import Image, display
display(Image(str(C.FIGURE_DIR / "class_imbalance.png")))

## Stage 2 - preprocessing

Fundus images arrive at different resolutions, aspect ratios and exposures.
Four operations normalise them:

1. **Black border crop** - trim the uninformative frame.
2. **Square resize** to 456px, cached to disk.
3. **Ben Graham enhancement** - subtract a heavily blurred copy of the image
   from itself, cancelling per-camera colour cast and uneven illumination so
   microaneurysms, haemorrhages and exudates stand out.
4. **Circular mask** - zero the corners so the network cannot key on
   camera-specific artefacts.

In [ ]:
from src.eda import preprocessing_steps
preprocessing_steps(df)
plt.close("all")
display(Image(str(C.FIGURE_DIR / "preprocessing_steps.png")))

### Samples at every grade

The visual difference between grade 0 and grade 4 is obvious; between
adjacent grades it is subtle, which is where most model errors land.

In [ ]:
from src.eda import samples_per_grade
samples_per_grade(df, n=4)
plt.close("all")
display(Image(str(C.FIGURE_DIR / "samples_by_grade.png")))

## Stage 3 - the split

A stratified 70/15/15 split keeps each grade's proportion identical across
the three sets. With only ~193 grade-3 images, a random split would leave
wildly different balances and make the validation curve unreadable.

The test split is held out entirely: no training, no model selection, no
threshold tuning ever touches it.

In [ ]:
splits = pd.read_csv(C.SPLITS_CSV)
pivot = splits.groupby(["split", "label"]).size().unstack(fill_value=0)
pivot = pivot.reindex(["train", "val", "test"])
pivot.columns = [C.CLASS_SHORT[c] for c in pivot.columns]
pivot["total"] = pivot.sum(axis=1)
pivot

## Stages 4-6 - the CNN, feature extraction and classification

Two model families, so the results chapter can quantify what transfer
learning is actually worth:

- **`baseline`** - a 5-block VGG-style CNN trained from scratch. No external
  data, ~4.9M parameters.
- **`efficientnet`** - EfficientNet-B3 pretrained on ImageNet, fine-tuned end
  to end. The backbone produces the stage-5 feature vector; a fresh linear
  head performs stage-6 classification.

In [ ]:
from src.models import build_model, count_parameters, get_device

device = get_device()
for key in ["baseline", "efficientnet"]:
    model = build_model(key, pretrained=False)
    total, trainable = count_parameters(model)
    cfg = C.MODELS[key]
    print(f"{key:14} {cfg['arch']:18} {total/1e6:6.2f}M params  "
          f"input {cfg['img_size']}px  batch {cfg['batch_size']}")

### Training

Training is done from the command line so it survives notebook restarts:

```bash
python -m src.train --model baseline
python -m src.train --model efficientnet
```

AdamW, cosine decay with a 2-epoch linear warmup, class-weighted
cross-entropy with label smoothing, gradient clipping, early stopping on
validation QWK. Roughly 17 min and 48 min respectively on an Apple M5.

In [ ]:
hist_path = C.LOG_DIR / "efficientnet_history.csv"
if hist_path.exists():
    hist = pd.read_csv(hist_path)
    display(hist[["epoch", "train_loss", "val_loss", "val_accuracy",
                  "val_qwk", "val_referable_sensitivity"]].round(4))
    best = hist.loc[hist["val_qwk"].idxmax()]
    print(f"\nBest epoch {int(best['epoch'])}: "
          f"val QWK {best['val_qwk']:.4f}, acc {best['val_accuracy']:.4f}")
else:
    print("No training history yet. Run:  python -m src.train --model efficientnet")

## Stage 7 - evaluation and prediction

Quadratic weighted kappa is the headline metric. Unlike accuracy it
penalises a grade 0 -> 4 error far more heavily than a grade 3 -> 4 error,
matching how a clinician judges mistakes.

We also collapse the five grades into the binary screening decision that a
programme actually acts on: **referable DR** is grade >= 2. Its sensitivity is
the number a clinical reader cares about most, because missing a severe case
is far worse than over-referring a mild one.

In [ ]:
import json

res_path = C.LOG_DIR / "efficientnet_test_results.json"
if res_path.exists():
    res = json.loads(res_path.read_text())
    m = res["metrics"]
    print(f"Test set (n={res['n_samples']})\n" + "-" * 40)
    for k in ["accuracy", "qwk", "f1_macro", "f1_weighted"]:
        print(f"  {k:24} {m[k]:.4f}")
    print("\n  Referable DR (grade >= 2)")
    for k in ["referable_sensitivity", "referable_specificity", "referable_precision"]:
        print(f"  {k:24} {m[k]:.4f}")

    display(Image(str(C.FIGURE_DIR / "efficientnet_confusion_matrix.png")))
else:
    print("No results yet. Run:  python -m src.evaluate --model efficientnet")

### Model comparison

The gap between the from-scratch baseline and the pretrained model is the
central quantitative claim of the project.

In [ ]:
cmp_path = C.LOG_DIR / "model_comparison_test.csv"
if cmp_path.exists():
    display(pd.read_csv(cmp_path).round(4))
    display(Image(str(C.FIGURE_DIR / "model_comparison_test.png")))
else:
    print("Run:  python -m src.compare")

### Explaining a single prediction

Grad-CAM backpropagates the predicted grade's score to the last
convolutional feature map, so the heatmap shows which retinal regions pushed
the prediction up. This turns "the model says grade 3" into "the model says
grade 3 because of these lesions" - the difference between a black box and
something a clinician can argue with.

In [ ]:
from src.predict import predict_file, available_models

if available_models():
    test_ids = splits[splits["split"] == "test"]["id_code"].tolist()
    sample = C.IMAGES_RAW / f"{test_ids[0]}.png"
    truth = splits.loc[splits["id_code"] == test_ids[0], "label"].iloc[0]

    r = predict_file(sample, model_key=available_models()[0])

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    import cv2
    raw = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
    for ax, img, title in [
        (axes[0], raw, "Original"),
        (axes[1], r["processed"], "Preprocessed"),
        (axes[2], r["overlay"], "Grad-CAM"),
    ]:
        ax.imshow(img); ax.set_title(title); ax.axis("off")
    plt.suptitle(f"True: {C.CLASS_NAMES[truth]}   |   "
                 f"Predicted: {r['label']} ({r['confidence']:.1%})",
                 fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

    print("Per-grade probabilities:")
    for g, p in enumerate(r["probabilities"]):
        print(f"  {C.CLASS_NAMES[g]:24} {p:6.2%}")
else:
    print("Train a model first.")

## Live demo

```bash
streamlit run app/app.py
```

Upload any fundus image and get the grade, per-class confidence, the
referable-DR decision, and the Grad-CAM overlay.

---

**Disclaimer.** Research prototype for academic assessment. Not a medical
device and not validated for clinical use.